# Feature Selection

## 1. Import libraries and dataset

In [1]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, chi2, RFE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
 
warnings.filterwarnings('ignore')

In [2]:
dataset=pd.read_csv("df.csv")

In [3]:
dataset.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,AvgMonthlySpend,NumServices
0,Female,No,Yes,No,1,No,No,DSL,No,Yes,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,14.92,1
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,...,No,No,One year,No,Mailed check,56.95,1889.50,No,53.99,3
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,36.05,3
3,Male,No,No,No,45,No,No,DSL,Yes,No,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,40.02,3
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,50.55,1


In [4]:
# Encoding target variable
dataset['Churn'] = dataset['Churn'].map({'Yes':1, 'No':0})

In [5]:
# One-hot encode categorical columns
dataset = pd.get_dummies(dataset, drop_first=True)

In [6]:
dataset

,tenure,MonthlyCharges,TotalCharges,Churn,AvgMonthlySpend,NumServices,gender_Male,SeniorCitizen_Yes,Partner_Yes,Dependents_Yes,...,DeviceProtection_Yes,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,29.85,29.85,0,14.92,1,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
1,34,56.95,1889.50,0,53.99,3,True,False,False,False,...,True,False,False,False,True,False,False,False,False,True
2,2,53.85,108.15,1,36.05,3,True,False,False,False,...,False,False,False,False,False,False,True,False,False,True
3,45,42.30,1840.75,0,40.02,3,True,False,False,False,...,True,True,False,False,True,False,False,False,False,False
4,2,70.70,151.65,1,50.55,1,False,False,False,False,...,False,False,False,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,24,84.80,1990.50,0,79.62,7,True,False,True,True,...,True,True,True,True,True,False,True,False,False,True
7039,72,103.20,7362.90,0,100.86,6,False,False,True,True,...,True,False,True,True,True,False,True,True,False,False
7040,11,29.60,346.45,0,28.87,1,False,False,True,True,...,False,False,False,False,False,False,True,False,True,False
7041,4,74.40,306.60,1,61.32,2,True,True,True,False,...,False,False,False,False,False,False,True,False,False,True


In [7]:
# Converting all True/False columns into 0 and 1 while keeping other columns unchanged for consistency across machine learning models.

# apply(...) --> Go through each column one by one
# lambda x: --> For each column (call it x), do this
# if x.dtype == 'bool' --> Check if column is True/False type
# x.astype(int) --> Convert True → 1 and False → 0
# else x --> If not boolean, leave it as it is

dataset = dataset.apply(lambda x: x.astype(int) if x.dtype == 'bool' else x) 
dataset

,tenure,MonthlyCharges,TotalCharges,Churn,AvgMonthlySpend,NumServices,gender_Male,SeniorCitizen_Yes,Partner_Yes,Dependents_Yes,...,DeviceProtection_Yes,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,29.85,29.85,0,14.92,1,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0
1,34,56.95,1889.50,0,53.99,3,1,0,0,0,...,1,0,0,0,1,0,0,0,0,1
2,2,53.85,108.15,1,36.05,3,1,0,0,0,...,0,0,0,0,0,0,1,0,0,1
3,45,42.30,1840.75,0,40.02,3,1,0,0,0,...,1,1,0,0,1,0,0,0,0,0
4,2,70.70,151.65,1,50.55,1,0,0,0,0,...,0,0,0,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,24,84.80,1990.50,0,79.62,7,1,0,1,1,...,1,1,1,1,1,0,1,0,0,1
7039,72,103.20,7362.90,0,100.86,6,0,0,1,1,...,1,0,1,1,1,0,1,1,0,0
7040,11,29.60,346.45,0,28.87,1,0,0,1,1,...,0,0,0,0,0,0,1,0,1,0
7041,4,74.40,306.60,1,61.32,2,1,1,1,0,...,0,0,0,0,0,0,1,0,0,1


## 2. Input & Output variable split

In [8]:
indep_X = dataset.drop(columns=['Churn'])    
dep_Y=dataset['Churn'] 

## 3. Defining functions

In [9]:
def get_models():
    return {
        'Logistic'  : LogisticRegression(random_state=0, max_iter=1000),
        'SVM_Linear': SVC(kernel='linear', random_state=0),
        'SVM_RBF'   : SVC(kernel='rbf',    random_state=0),
        'KNN'       : KNeighborsClassifier(n_neighbors=5),
        'NaiveBayes': GaussianNB(),
        'Decision'  : DecisionTreeClassifier(criterion='entropy', random_state=0),
        'RandomForest': RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0),
    }
 
# Split + Scale
def split_and_scale(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
    
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test  = sc.transform(X_test)
    return X_train, X_test, y_train, y_test
 
# Train all 7 models and return accuracy dictionary 
def train_all_models(X_train, X_test, y_train, y_test):
    results = {}
    for name, model in get_models().items():
        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))
        results[name] = round(acc, 4)
    return results

## 4. SelectKbest method

In [10]:
print("SELECTKBEST - Accuracy for each k  (Chi-Square scoring)")
print("=" * 65)

k_values   = [5, 7, 10, 12]    # List of different k values (number of features to select)      
skb_results = {}               # Dictionary to store results for each k
 
for k in k_values:      # Loop through each k value
    selector   = SelectKBest(score_func=chi2, k=k)     # Select top k important features using Chi-Square method
    X_selected = selector.fit_transform(indep_X, dep_Y)    # Fit on data and transform to keep only selected features
 
     # Get the names of selected features
    selected_cols = indep_X.columns[selector.get_support()]

     # Split data into train and test, then apply scaling
    X_train, X_test, y_train, y_test = split_and_scale(X_selected, dep_Y)

     # Train all models and store their accuracy results
    skb_results[f'k={k}'] = train_all_models(X_train, X_test, y_train, y_test)

     # Store the selected feature names for this k
    skb_results[f'k={k}']['Selected_Features'] = list(selected_cols)   
 
# Creating a comparison table (only accuracy values, excluding feature list)
# Convert results dictionary into a clean table (rows = k values, columns = models, values = accuracy)
skb_table = pd.DataFrame(
    {k: {m: v for m, v in scores.items() if m != 'Selected_Features'}  # m-> model name, v-> accuracy value
     for k, scores in skb_results.items()}).T           # .T-> Transpose (swap rows & columns) 

# Print the comparison table
print(skb_table.to_string())
print()
 
# Print selected features for each k value
print("── Features selected by SelectKBest ──")
for k, scores in skb_results.items():
    print(f"  {k}: {scores['Selected_Features']}")
print()

SELECTKBEST - Accuracy for each k  (Chi-Square scoring)
      Logistic  SVM_Linear  SVM_RBF     KNN  NaiveBayes  Decision  RandomForest
k=5     0.7785      0.7763   0.7814  0.7626      0.6882    0.7161        0.7581
k=7     0.7905      0.7785   0.7899  0.7638      0.6871    0.7138        0.7530
k=10    0.7933      0.7842   0.7978  0.7723      0.7303    0.7342        0.7694
k=12    0.7893      0.7859   0.7893  0.7660      0.7354    0.7337        0.7694

── Features selected by SelectKBest ──
  k=5: ['tenure', 'MonthlyCharges', 'TotalCharges', 'Contract_Two year', 'PaymentMethod_Electronic check']
  k=7: ['tenure', 'MonthlyCharges', 'TotalCharges', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_Two year', 'PaymentMethod_Electronic check']
  k=10: ['tenure', 'MonthlyCharges', 'TotalCharges', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_Yes', 'TechSupport_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Electronic check']
  k=12: ['tenu

<h5 style="color:darkblue;"> 
• From the SelectKBest results, different numbers of features (k = 5, 7, 10, 12) have been tested to see which gives the best model performance.
<br> <br>

• When k = 10, most models achieved their highest accuracy, especially SVM (RBF), which performed the best overall. This shows that using 10 features gives a good balance between keeping important information and removing unnecessary features.<br> 

• The selected features mainly include tenure, monthly and total charges, contract type, internet service type and payment method, which clearly influence customer churn.<br> 

• Increasing features beyond 10 did not improve performance much, so k = 10 is selected as the optimal number of features.
</h5>

## 5. RFE Method

In [12]:
# SVC(linear) and RandomForest are not used here - they are very slow inside RFE
# because RFE retrains the model once per feature eliminated.
# LogReg and DT are much faster and still give good feature rankings.

rfe_estimators = {                                                        # Define models to use inside RFE (faster models for feature ranking)
    'LogReg': LogisticRegression(max_iter=1000, random_state=0),
    'DT'    : DecisionTreeClassifier(random_state=0)}    
 
# Dictionary to store results for each k and each estimator
rfe_results = {}
 
for k in k_values:
    rfe_results[k] = {}
    for est_name, estimator in rfe_estimators.items():
        rfe        = RFE(estimator=estimator, n_features_to_select=k, step=2)  # step=2 eliminates 2 features at a time → 2x faster
        X_selected = rfe.fit_transform(indep_X, dep_Y)
 
        selected_cols = indep_X.columns[rfe.get_support()]
 
        X_train, X_test, y_train, y_test = split_and_scale(X_selected, dep_Y)
        accs = train_all_models(X_train, X_test, y_train, y_test)
        accs['Selected_Features'] = list(selected_cols)
        rfe_results[k][est_name] = accs    # Save results for this k and estimator
 
# Formatting output for better readability
SEP  = "=" * 75
SEP2 = "-" * 75

# Loop through each k to print results
for k in k_values:
    print(f"\n{SEP}")
    print(f"  RFE  |  k = {k} features selected")
    print(SEP)
 
    # Create table of accuracies (exclude feature list)
    tbl = pd.DataFrame(
        {est: {m: f"{v:.4f}" for m, v in scores.items() if m != 'Selected_Features'}
         for est, scores in rfe_results[k].items()}).T
    
    tbl.index.name = "RFE Estimator"
    print(tbl.to_string())
 
 # Print selected features clearly
    print(f"\n  {'─'*35}")
    print(f"  Features Selected (k={k})")
    print(f"  {'─'*35}")
    
    for est, scores in rfe_results[k].items():
        feats = scores['Selected_Features']
        print(f"\n  [{est}]")
        for i, f in enumerate(feats, 1):
            print(f"    {i:>2}. {f}")
 
print(f"\n{SEP2}\n")


  RFE  |  k = 5 features selected
              Logistic SVM_Linear SVM_RBF     KNN NaiveBayes Decision RandomForest
RFE Estimator                                                                     
LogReg          0.7717     0.7547  0.7717  0.7087     0.7217   0.7717       0.7717
DT              0.7865     0.7848  0.7842  0.7609     0.7314   0.7093       0.7479

  ───────────────────────────────────
  Features Selected (k=5)
  ───────────────────────────────────

  [LogReg]
     1. InternetService_Fiber optic
     2. InternetService_No
     3. OnlineSecurity_Yes
     4. Contract_One year
     5. Contract_Two year

  [DT]
     1. tenure
     2. MonthlyCharges
     3. TotalCharges
     4. AvgMonthlySpend
     5. InternetService_Fiber optic

  RFE  |  k = 7 features selected
              Logistic SVM_Linear SVM_RBF     KNN NaiveBayes Decision RandomForest
RFE Estimator                                                                     
LogReg          0.7740     0.7547  0.7740  0.752

<h5 style="color:darkblue;"> 
• RFE using Decision Tree gives better feature selection compared to Logistic Regression.<br><br> 

• Best performance is achieved when selecting around 10–12 features. <br>

• Important features consistently include: Tenure, MonthlyCharges, TotalCharges, Contract type, Internet service, Payment method <br>

• This confirms that both customer duration, billing and service usage play a key role in predicting churn.
</h5>

## 6. Feature Importance method

In [13]:
# Random Forest feature_importances_ ranks features by contribution.

print("FEATURE IMPORTANCE (Random Forest) - Accuracy for each k")
print("=" * 65)
 
# Train one RF to get stable importance scores
rf_for_importance = RandomForestClassifier(n_estimators=100, random_state=0)
rf_for_importance.fit(indep_X, dep_Y)

# Create a DataFrame with feature names and their importance values, sorted descending
importance_df = pd.DataFrame({
    'Feature'   : indep_X.columns,
    'Importance': rf_for_importance.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

# Display top 10 most important features
print("\nTop 10 features by importance:")
print(importance_df.head(10).to_string(index=False))
print()

# Dictionary to store results for different k values
fi_results = {}

# Loop through different k values
for k in k_values:
    top_features   = importance_df['Feature'].head(k).tolist()   # Select top k important features
    X_selected     = indep_X[top_features]
 
    X_train, X_test, y_train, y_test = split_and_scale(X_selected, dep_Y)    # Split data and apply scaling
    fi_results[f'k={k}'] = train_all_models(X_train, X_test, y_train, y_test)   # Train all models and store accuracy
    fi_results[f'k={k}']['Selected_Features'] = top_features     # Store selected feature names

# Convert results into comparison table (exclude feature list)
fi_table = pd.DataFrame(
    {k: {m: v for m, v in scores.items() if m != 'Selected_Features'}
     for k, scores in fi_results.items()}).T

# Print accuracy table
print(fi_table.to_string())
print()

# Print selected features for each k

print("── Features selected by Feature Importance ──")

for k, scores in fi_results.items():
    print(f"  {k}: {scores['Selected_Features']}")

print()

FEATURE IMPORTANCE (Random Forest) - Accuracy for each k

Top 10 features by importance:
                       Feature  Importance
                  TotalCharges    0.163939
                MonthlyCharges    0.143985
                        tenure    0.137736
               AvgMonthlySpend    0.137544
   InternetService_Fiber optic    0.039991
PaymentMethod_Electronic check    0.038668
                   NumServices    0.034098
             Contract_Two year    0.033764
                   gender_Male    0.024558
          PaperlessBilling_Yes    0.023157

      Logistic  SVM_Linear  SVM_RBF     KNN  NaiveBayes  Decision  RandomForest
k=5     0.7865      0.7848   0.7842  0.7609      0.7314    0.7098        0.7507
k=7     0.7819      0.7825   0.7871  0.7729      0.7439    0.7325        0.7655
k=10    0.7825      0.7848   0.7859  0.7757      0.6968    0.7235        0.7780
k=12    0.7859      0.7888   0.7899  0.7751      0.7093    0.7405        0.7802

── Features selected by Feature Impo

<h5 style="color:darkblue;"> 
• Feature Importance method shows that billing-related features like TotalCharges, MonthlyCharges, tenure, and AvgMonthlySpend are the most influential in predicting churn. <br>  <br> 

• Model performance improves as more features are added, with best results observed at k = 12. <br> 

• SVM and Logistic Regression give the highest accuracy among all models. <br> 

• This confirms that both customer financial behavior and service usage play a key role in churn prediction. 
</h5>

## 7. Feature Selection - Summary table

In [14]:
# SUMMARY TABLE  (best accuracy across all methods)
# best method + k combination = the row with the highest accuracies across models  
    
print("=" * 65)
print("SUMMARY - Best accuracy per method (max over k values)")
print("=" * 65)
 
model_cols = list(get_models().keys())  # Get model names (columns)
 
summary_rows = {}   # Dictionary to store best results for each method
 
# SelectKBest best row
summary_rows['SKB_best'] = skb_table[model_cols].max().to_dict()    # Take maximum accuracy for each model across all k values
summary_rows['SKB_best']['best_k'] = skb_table[model_cols].mean(axis=1).idxmax()  # Find k value with highest average performance
 
# Flatten RFE results into one table (combine all k + estimators)
rfe_flat_rows = {}

for k in k_values:
    for est in rfe_estimators:
        label = f"RFE_k{k}_{est}"  # label like RFE_k10_LogReg
        rfe_flat_rows[label] = {m: rfe_results[k][est][m] for m in model_cols}

# Convert to DataFrame
rfe_flat_df = pd.DataFrame(rfe_flat_rows).T

# Take best accuracy across all combinations
summary_rows['RFE_best'] = rfe_flat_df[model_cols].max().to_dict()

# Find best combination (k + estimator)
summary_rows['RFE_best']['best_k'] = rfe_flat_df[model_cols].mean(axis=1).idxmax()
 
# Feature Importance best row
summary_rows['FI_best'] = fi_table[model_cols].max().to_dict()  # Take best accuracy across k values
summary_rows['FI_best']['best_k'] = fi_table[model_cols].mean(axis=1).idxmax()  # Find best k

# Convert everything into final summary table
grand_summary = pd.DataFrame(summary_rows).T

print(grand_summary.to_string())  # Print final comparison
print()

SUMMARY - Best accuracy per method (max over k values)
         Logistic SVM_Linear SVM_RBF     KNN NaiveBayes Decision RandomForest      best_k
SKB_best   0.7933     0.7859  0.7978  0.7723     0.7354   0.7342       0.7694        k=10
RFE_best   0.7865     0.7882  0.7882    0.77     0.7433    0.774        0.774  RFE_k12_DT
FI_best    0.7865     0.7888  0.7899  0.7757     0.7439   0.7405       0.7802        k=12



<h5 style="color:darkblue;"> 
• Among all feature selection methods, SelectKBest with k = 10 features gives the best performance, especially with SVM (RBF kernel).<br><br>

• Feature Importance and RFE also provide competitive results, with optimal performance around 10–12 features.<br>

• Overall, SVM and Logistic Regression models consistently perform better across different feature selection techniques.<br>

• This shows that selecting the right number of features (around 10–12) significantly improves model performance.
</h5>

## 8. Selecting Best Features

In [15]:
selector   = SelectKBest(score_func=chi2, k=10)   # best method + k from summary
X_best     = selector.fit_transform(indep_X, dep_Y)
 
X_train, X_test, y_train, y_test = train_test_split(X_best, dep_Y, test_size=0.25, random_state=0)
sc      = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test  = sc.transform(X_test)

# Model Creation

In [18]:
# Defining Models with their hyperparameter grids

models_params = {'Logistic Regression': {'model' : LogisticRegression(random_state=0, max_iter=1000),
                                         'params': {'C': [0.01, 0.1, 1, 10], 'penalty': ['l2'], 'solver' : ['lbfgs', 'liblinear']}},
 
'SVM (RBF)': {'model' : SVC(random_state=0),'params': {'C': [0.1, 1, 10],'gamma': ['scale', 'auto'],'kernel': ['rbf']}},
 
'SVM (Linear)': {'model' : SVC(random_state=0),'params': {'C'     : [0.01, 0.1, 1, 10],'kernel': ['linear']}},
 
'KNN': {'model' : KNeighborsClassifier(),'params': {'n_neighbors': [3, 5, 7, 9], 'metric': ['minkowski', 'euclidean']}},
 
'Naive Bayes': {'model' : GaussianNB(),'params': {'var_smoothing': [1e-9, 1e-8, 1e-7]}},
 
'Decision Tree': {'model' : DecisionTreeClassifier(random_state=0),'params': {'criterion': ['entropy', 'gini'],'max_depth': [None, 5, 10],
                                                                              'min_samples_split': [2, 5]}},
 
'Random Forest': {'model' : RandomForestClassifier(random_state=0),'params': {'n_estimators': [50, 100],'criterion': ['entropy', 'gini'],
                                                                              'max_depth'   : [None, 5, 10]}}}

In [19]:
# Run Grid Search on all models
results = []
 
for name, mp in models_params.items():  # Loop through each model and its parameter grid
    
    # Create GridSearchCV object to tune hyperparameters using cross-validation
    gs = GridSearchCV(estimator  = mp['model'], param_grid = mp['params'], cv = 5, scoring = 'accuracy', n_jobs = -1)          

    # Fit the model on training data to find the best parameters
    gs.fit(X_train, y_train)

    # Predict the target values using the best model on test data
    y_pred    = gs.best_estimator_.predict(X_test)

    # Calculate accuracy of the model on test data
    test_acc  = accuracy_score(y_test, y_pred)

    # Store model name, best parameters, and accuracy results
    results.append({
        'Model'        : name,
        'Best Params'  : gs.best_params_,
        'CV Accuracy'  : round(gs.best_score_, 4),
        'Test Accuracy': round(test_acc, 4)})

    # Print model name in formatted style
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")

    
    print(f"  Best params   : {gs.best_params_}")      # Print best hyperparameters found
    print(f"  CV accuracy   : {gs.best_score_:.4f}")   # Print cross-validation accuracy score
    print(f"  Test accuracy : {test_acc:.4f}")         # Print test accuracy score

    # Print detailed classification metrics (precision, recall, F1-score)
    print(f"\n{classification_report(y_test, y_pred, target_names=['No Churn', 'Churn'])}")
    


  Logistic Regression
  Best params   : {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
  CV accuracy   : 0.7986
  Test accuracy : 0.7933

              precision    recall  f1-score   support

    No Churn       0.84      0.89      0.86      1298
       Churn       0.63      0.52      0.57       463

    accuracy                           0.79      1761
   macro avg       0.73      0.70      0.72      1761
weighted avg       0.78      0.79      0.79      1761


  SVM (RBF)
  Best params   : {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
  CV accuracy   : 0.8020
  Test accuracy : 0.7927

              precision    recall  f1-score   support

    No Churn       0.84      0.89      0.86      1298
       Churn       0.63      0.52      0.57       463

    accuracy                           0.79      1761
   macro avg       0.73      0.71      0.72      1761
weighted avg       0.78      0.79      0.79      1761


  SVM (Linear)
  Best params   : {'C': 0.01, 'kernel': 'linear'}
  CV accur

<h5 style="color:darkblue;"> 
• All models achieved similar accuracy around 79%. <br><br> 
• Logistic Regression and SVM (RBF) performed best overall. <br><br>  
• However, the model struggles to correctly identify churn customers due to class imbalance, as seen from lower recall values. <br><br>  
• Naive Bayes achieved higher churn recall but at the cost of lower overall accuracy. <br><br>  
• Therefore, Logistic Regression is selected as the final model due to its balanced and stable performance.
</h5>